# JOB MARKET DATA CLEANING
## Focused Dataset Preparation for Machine Learning

## 1. Setup & Configuration

In [15]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuration
INPUT_FILE = 'job_descriptions_100k_random_sample.csv'
OUTPUT_FILE = 'jobs_cleaned_for_prediction.csv'

print("Job Prediction Dataset Cleaning")
print("Goal: Keep ONLY prediction-relevant features\n")

Job Prediction Dataset Cleaning
Goal: Keep ONLY prediction-relevant features



## 2. Load Data

In [16]:
# Load dataset
print("Loading dataset...")
df = pd.read_csv('jobs.csv', low_memory=False)
print(f" Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

print("Original Columns:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")

Loading dataset...
 Loaded: 99,636 rows × 25 columns

Original Columns:
    1. Job Id
    2. Qualifications
    3. location
    4. Country
    5. latitude
    6. longitude
    7. Work Type
    8. Company Size
    9. Job Posting Date
   10. Preference
   11. Contact Person
   12. Contact
   13. Job Title
   14. Role
   15. Job Portal
   16. Job Description
   17. Benefits
   18. skills
   19. Responsibilities
   20. Company
   21. Company Profile
   22. Min_Salary_K
   23. Max_Salary_K
   24. Min_Experience
   25. Max_Experience


In [17]:
print("\nREMOVING UNNECESSARY COLUMNS")

# Define columns to REMOVE (not useful for prediction)
columns_to_remove = [
    # Identifiers (not predictive)
    'Job Id',
    
    # Geographic details (Country is enough)
    'location',
    'latitude',
    'longitude',
    
    # Time data (not predictive of salary)
    'Job Posting Date',
    
    # Not useful for prediction
    'Preference',           # Gender preference
    'Contact Person',       # Personal info
    'Contact',             # Personal info
    
    # Complex/Text data (requires separate NLP processing)
    'Benefits',            # Nested complex data
    'Job Description',     # Long text (use NLP separately)
    'Responsibilities',    # Long text (use NLP separately)
    
    # Too many unique values (poor predictor)
    'Company',             # 885 unique companies
    'Company Profile'      # Nested complex data
]

# Create cleaned dataset
df_cleaned = df.copy()

# Remove columns
removed_count = 0
for col in columns_to_remove:
    if col in df_cleaned.columns:
        df_cleaned = df_cleaned.drop(columns=[col])
        removed_count += 1
        print(f"  Removed: {col}")

print(f"\n Removed {removed_count} columns")
print(f"Remaining: {df_cleaned.shape[1]} columns (focused for prediction)")


REMOVING UNNECESSARY COLUMNS
  Removed: Job Id
  Removed: location
  Removed: latitude
  Removed: longitude
  Removed: Job Posting Date
  Removed: Preference
  Removed: Contact Person
  Removed: Contact
  Removed: Benefits
  Removed: Job Description
  Removed: Responsibilities
  Removed: Company
  Removed: Company Profile

 Removed 13 columns
Remaining: 12 columns (focused for prediction)


## 4. Data Cleaning

In [18]:
print("\nDATA CLEANING")
print("="*60)

initial_rows = len(df_cleaned)

# 1. Remove duplicates
print("\n1. Removing duplicates...")
before = len(df_cleaned)
df_cleaned = df_cleaned.drop_duplicates()
removed = before - len(df_cleaned)
print(f"   Removed: {removed:,} duplicate rows")

# 2. Handle missing values (there should be minimal)
print("\n2. Checking missing values...")
missing = df_cleaned.isnull().sum()
if missing.sum() > 0:
    print("   Found missing values:")
    print(missing[missing > 0])
    df_cleaned = df_cleaned.dropna()
    print(f"   Dropped rows with missing values")
else:
    print("  No missing values")

print(f"\n Data retained: {len(df_cleaned):,} / {initial_rows:,} rows ({len(df_cleaned)/initial_rows*100:.1f}%)")


DATA CLEANING

1. Removing duplicates...
   Removed: 0 duplicate rows

2. Checking missing values...
  No missing values

 Data retained: 99,636 / 99,636 rows (100.0%)


## 5. Feature Transformation

In [19]:
print("\nFEATURE TRANSFORMATION")

# 1. Extract Salary Components
print("\n Processing Salary Range...")
if 'Salary Range' in df_cleaned.columns:
    salary_split = df_cleaned['Salary Range'].str.split('-', expand=True)
    
    df_cleaned['Min_Salary_K'] = (
        salary_split[0]
        .str.replace('$', '', regex=False)
        .str.replace('K', '', regex=False)
        .astype(int)
    )
    
    df_cleaned['Max_Salary_K'] = (
        salary_split[1]
        .str.replace('$', '', regex=False)
        .str.replace('K', '', regex=False)
        .astype(int)
    )
    
    # Drop original
    df_cleaned = df_cleaned.drop(columns=['Salary Range'])
    print(f"   Created Min_Salary_K and Max_Salary_K")
    print(f"   Dropped original Salary Range column")

# 2. Extract Experience Components
print("\nProcessing Experience...")
if 'Experience' in df_cleaned.columns:
    exp_split = df_cleaned['Experience'].str.split(' to ', expand=True)
    
    df_cleaned['Min_Experience'] = (
        exp_split[0]
        .str.extract(r'(\d+)')[0]
        .astype(int)
    )
    
    df_cleaned['Max_Experience'] = (
        exp_split[1]
        .str.replace(' Years', '', regex=False)
        .astype(int)
    )
    
    # Drop original
    df_cleaned = df_cleaned.drop(columns=['Experience'])
    print(f"   Created Min_Experience and Max_Experience")
    print(f"   Dropped original Experience column")


FEATURE TRANSFORMATION

 Processing Salary Range...

Processing Experience...


## 6. Final Dataset Structure

In [20]:
print("\nFINAL DATASET STRUCTURE")

# Reorder columns for logical structure
column_order = [
    # Target variables (what we predict)
    'Min_Salary_K',
    'Max_Salary_K',
    
    # Job characteristics
    'Job Title',
    'Role',
    'Qualifications',
    'Min_Experience',
    'Max_Experience',
    'skills',
    
    # Employment details
    'Work Type',
    'Company Size',
    
    # Location
    'Country',
    
    # Source
    'Job Portal'
]

# Keep only existing columns in desired order
existing_order = [col for col in column_order if col in df_cleaned.columns]
df_cleaned = df_cleaned[existing_order]

print("\nFINAL COLUMNS (Prediction-Ready):")
print("-" * 60)
for i, col in enumerate(df_cleaned.columns, 1):
    dtype = df_cleaned[col].dtype
    unique = df_cleaned[col].nunique()
    print(f"{i:2d}. {col:20s} | Type: {str(dtype):10s} | Unique: {unique:6d}")

print(f"\nFinal Dataset: {df_cleaned.shape[0]:,} rows × {df_cleaned.shape[1]} columns")


FINAL DATASET STRUCTURE

FINAL COLUMNS (Prediction-Ready):
------------------------------------------------------------
 1. Min_Salary_K         | Type: int64      | Unique:     11
 2. Max_Salary_K         | Type: int64      | Unique:     51
 3. Job Title            | Type: object     | Unique:    147
 4. Role                 | Type: object     | Unique:    376
 5. Qualifications       | Type: object     | Unique:     10
 6. Min_Experience       | Type: int64      | Unique:      6
 7. Max_Experience       | Type: int64      | Unique:      8
 8. skills               | Type: object     | Unique:    376
 9. Work Type            | Type: object     | Unique:      5
10. Company Size         | Type: int64      | Unique:  68088
11. Country              | Type: object     | Unique:    216
12. Job Portal           | Type: object     | Unique:     16

Final Dataset: 99,636 rows × 12 columns


## 7. Memory Optimization

In [21]:
print("\nMEMORY OPTIMIZATION")

memory_before = df_cleaned.memory_usage(deep=True).sum() / 1024**2
print(f"\nMemory before: {memory_before:.2f} MB")

# Optimize categorical columns
categorical_cols = ['Work Type', 'Qualifications', 'Job Portal', 'Country']
for col in categorical_cols:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].astype('category')
        print(f"   ✓ {col} → category")

# Optimize integer columns
int_cols = ['Min_Salary_K', 'Max_Salary_K', 'Min_Experience', 'Max_Experience', 'Company Size']
for col in int_cols:
    if col in df_cleaned.columns:
        col_min = df_cleaned[col].min()
        col_max = df_cleaned[col].max()
        
        if col_min >= 0 and col_max <= 255:
            df_cleaned[col] = df_cleaned[col].astype('uint8')
            print(f"   ✓ {col} → uint8")
        elif col_min >= 0 and col_max <= 65535:
            df_cleaned[col] = df_cleaned[col].astype('uint16')
            print(f"   ✓ {col} → uint16")
        elif col_min >= 0:
            df_cleaned[col] = df_cleaned[col].astype('uint32')
            print(f"   ✓ {col} → uint32")

memory_after = df_cleaned.memory_usage(deep=True).sum() / 1024**2
savings = memory_before - memory_after

print(f"\nMemory after: {memory_after:.2f} MB")
print(f"Saved: {savings:.2f} MB ({savings/memory_before*100:.1f}% reduction)")


MEMORY OPTIMIZATION

Memory before: 56.21 MB
   ✓ Work Type → category
   ✓ Qualifications → category
   ✓ Job Portal → category
   ✓ Country → category
   ✓ Min_Salary_K → uint8
   ✓ Max_Salary_K → uint8
   ✓ Min_Experience → uint8
   ✓ Max_Experience → uint8
   ✓ Company Size → uint32

Memory after: 32.06 MB
Saved: 24.15 MB (43.0% reduction)


## 8. Data Validation

In [22]:
print("\nDATA VALIDATION")

validation_passed = True

# 1. Check for missing values
print("\n1. Missing Values Check:")
missing = df_cleaned.isnull().sum().sum()
if missing == 0:
    print("  No missing values")
else:
    print(f"  Found {missing} missing values")
    validation_passed = False

# 2. Check salary ranges
print("\n2. Salary Range Validation:")
invalid_salary = (df_cleaned['Min_Salary_K'] > df_cleaned['Max_Salary_K']).sum()
if invalid_salary == 0:
    print("   All salary ranges valid (Min ≤ Max)")
    print(f"   Range: ${df_cleaned['Min_Salary_K'].min()}K - ${df_cleaned['Max_Salary_K'].max()}K")
else:
    print(f"    Found {invalid_salary} invalid salary ranges")
    validation_passed = False

# 3. Check experience ranges
print("\n3. Experience Range Validation:")
invalid_exp = (df_cleaned['Min_Experience'] > df_cleaned['Max_Experience']).sum()
if invalid_exp == 0:
    print("   All experience ranges valid (Min ≤ Max)")
    print(f"   Range: {df_cleaned['Min_Experience'].min()} - {df_cleaned['Max_Experience'].max()} years")
else:
    print(f"   Found {invalid_exp} invalid experience ranges")
    validation_passed = False

# 4. Check for duplicates
print("\n4. Duplicate Rows Check:")
dupes = df_cleaned.duplicated().sum()
if dupes == 0:
    print("   No duplicate rows")
else:
    print(f"  Found {dupes} duplicate rows")
    validation_passed = False

# 5. Check column count
print("\n5. Column Structure:")
print(f"   {df_cleaned.shape[1]} columns (streamlined for prediction)")
print(f"   {df_cleaned.shape[0]:,} rows ready for modeling")

if validation_passed:
    print("ALL VALIDATION CHECKS PASSED")
else:
    print(" VALIDATION FAILED - Review errors above")


DATA VALIDATION

1. Missing Values Check:
  No missing values

2. Salary Range Validation:
   All salary ranges valid (Min ≤ Max)
   Range: $55K - $130K

3. Experience Range Validation:
   All experience ranges valid (Min ≤ Max)
   Range: 0 - 15 years

4. Duplicate Rows Check:
   No duplicate rows

5. Column Structure:
   12 columns (streamlined for prediction)
   99,636 rows ready for modeling
ALL VALIDATION CHECKS PASSED


## 9. Export Clean Dataset

In [25]:
print("\n EXPORTING CLEAN DATASET")

# Save main dataset
print(f"\nSaving to: {OUTPUT_FILE}")
df_cleaned.to_csv(OUTPUT_FILE, index=False)
print("Main dataset saved")

# Create and save data dictionary
dict_filename = OUTPUT_FILE.replace('.csv', '_dictionary.csv')
data_dict = pd.DataFrame({
    'Column': df_cleaned.columns,
    'Data_Type': df_cleaned.dtypes.astype(str),
    'Non_Null_Count': df_cleaned.count().values,
    'Unique_Values': [df_cleaned[col].nunique() for col in df_cleaned.columns],
    'Sample_Value': [str(df_cleaned[col].iloc[0]) for col in df_cleaned.columns]
})
data_dict.to_csv(dict_filename, index=False)
print(f"Data dictionary saved: {dict_filename}")

# Summary
print(" CLEANING SUMMARY")
print(f"\nColumns Removed: {df.shape[1] - df_cleaned.shape[1]}")
print(f"   • Geographic: latitude, longitude, location")
print(f"   • Personal: Contact Person, Contact, Preference")
print(f"   • Text Data: Job Description, Responsibilities, Benefits")
print(f"   • Identifiers: Job Id")
print(f"   • Company: Company, Company Profile")
print(f"   • Time: Job Posting Date")

print(f"\nColumns Retained: {df_cleaned.shape[1]}")
print(f"   • Target: Min/Max_Salary_K")
print(f"   • Features: Job Title, Role, Qualifications, Experience, etc.")

print(f"\nData Quality:")
print(f"   • Rows: {df_cleaned.shape[0]:,}")
print(f"   • Columns: {df_cleaned.shape[1]}")
print(f"   • Missing Values: 0")
print(f"   • Duplicates: 0")
print(f"   • Memory: {df_cleaned.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n Dataset Ready For:")
print(f"   • Salary prediction models")
print(f"   • Job matching algorithms")
print(f"   • Skill gap analysis")
print(f"   • Market trend analysis")

print(" DATA CLEANING COMPLETE!")


 EXPORTING CLEAN DATASET

Saving to: jobs_cleaned_for_prediction.csv
Main dataset saved
Data dictionary saved: jobs_cleaned_for_prediction_dictionary.csv
 CLEANING SUMMARY

Columns Removed: 13
   • Geographic: latitude, longitude, location
   • Personal: Contact Person, Contact, Preference
   • Text Data: Job Description, Responsibilities, Benefits
   • Identifiers: Job Id
   • Company: Company, Company Profile
   • Time: Job Posting Date

Columns Retained: 12
   • Target: Min/Max_Salary_K
   • Features: Job Title, Role, Qualifications, Experience, etc.

Data Quality:
   • Rows: 99,636
   • Columns: 12
   • Missing Values: 0
   • Duplicates: 0
   • Memory: 32.06 MB

 Dataset Ready For:
   • Salary prediction models
   • Job matching algorithms
   • Skill gap analysis
   • Market trend analysis
 DATA CLEANING COMPLETE!


## 10. Preview Cleaned Data

In [26]:
print("\nCLEANED DATASET PREVIEW")
print("\nFirst 10 rows:")
display(df_cleaned.head(10))

print("\nStatistical Summary:")
display(df_cleaned.describe())

print("\nCategorical Distribution:")
for col in ['Work Type', 'Qualifications', 'Country']:
    if col in df_cleaned.columns:
        print(f"\n{col}:")
        print(df_cleaned[col].value_counts().head())


CLEANED DATASET PREVIEW

First 10 rows:


,Min_Salary_K,Max_Salary_K,Job Title,Role,Qualifications,Min_Experience,Max_Experience,skills,Work Type,Company Size,Country,Job Portal
0,55,84,Procurement Manager,Supplier Diversity Manager,BBA,5,10,Supplier diversity programs Diversity and incl...,Contract,93242,Panama,The Muse
1,61,108,Architectural Designer,Architectural Drafter,MBA,0,12,Architectural drafting AutoCAD 2D and 3D model...,Part-Time,18411,Tunisia,Idealist
2,57,82,Art Teacher,Art Education Coordinator,M.Com,0,11,Art education curriculum Program development T...,Full-Time,120621,Zimbabwe,ZipRecruiter
3,56,95,Environmental Consultant,Environmental Impact Analyst,B.Com,5,12,Environmental impact analysis Data collection ...,Temporary,128908,Albania,Internships.com
4,58,122,Art Teacher,Art Education Coordinator,BCA,4,13,Art education curriculum Program development T...,Temporary,114717,Iraq,LinkedIn
5,62,83,Event Planner,Wedding Planner,MBA,5,10,Wedding planning Venue selection Catering and ...,Contract,100441,Andorra,ZipRecruiter
6,57,96,Architect,Architectural Designer,M.Com,3,11,"Architectural design software (e.g., AutoCAD, ...",Contract,85517,Eritrea,Dice
7,57,112,Family Lawyer,Mediator,BA,5,14,Mediation Conflict resolution Negotiation Comm...,Contract,81044,Haiti,USAJOBS
8,59,123,Account Manager,Client Relationship Manager,PhD,5,12,Client relationship management Customer servic...,Temporary,17650,North Korea,FlexJobs
9,56,114,Business Development Manager,Market Expansion Manager,BBA,4,8,Market expansion strategies Market research Sa...,Part-Time,60092,Uruguay,Idealist



Statistical Summary:


,Min_Salary_K,Max_Salary_K,Min_Experience,Max_Experience,Company Size
count,99636.000000,99636.000000,99636.000000,99636.000000,99636.000000
mean,60.003081,105.028223,2.502208,11.506333,73656.165894
std,3.166620,14.683930,1.706609,2.289319,35295.811639
min,55.000000,80.000000,0.000000,8.000000,12646.000000
25%,57.000000,92.000000,1.000000,10.000000,43098.750000
50%,60.000000,105.000000,3.000000,12.000000,73435.000000
75%,63.000000,118.000000,4.000000,14.000000,104276.000000
max,65.000000,130.000000,5.000000,15.000000,134834.000000



Categorical Distribution:

Work Type:
Work Type
Temporary    20050
Contract     20047
Part-Time    19901
Intern       19874
Full-Time    19764
Name: count, dtype: int64

Qualifications:
Qualifications
PhD       10115
B.Com     10051
B.Tech    10028
BA        10011
M.Com      9994
Name: count, dtype: int64

Country:
Country
Namibia             522
Cameroon            513
Guinea              513
Papua New Guinea    511
Sri Lanka           511
Name: count, dtype: int64
